# Melanoma Detection

In [1]:
# Imports
import os
import time
import shutil
import random
import kagglehub
import polars as pl
import torch
from torch import nn
import torchvision
import torchmetrics
from torcheval.metrics import MulticlassAccuracy

/home/cam/miniforge3/envs/jupyter_dl/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

## Preparing Data

In [3]:
# Download latest version
path = kagglehub.dataset_download("andrewmvd/isic-2019")

print("Path to dataset files:", path)

Path to dataset files: /home/cam/.cache/kagglehub/datasets/andrewmvd/isic-2019/versions/1


In [4]:
!ls {path}

ISIC_2019_Training_GroundTruth.csv  ISIC_2019_Training_Metadata.csv
ISIC_2019_Training_Input


In [5]:
df_meta = pl.read_csv(os.path.join(path, "ISIC_2019_Training_Metadata.csv"))
df_labels = pl.read_csv(os.path.join(path, "ISIC_2019_Training_GroundTruth.csv"))
df_meta.head(), df_labels.head()

(shape: (5, 5)
 ┌──────────────┬────────────┬─────────────────────┬───────────┬────────┐
 │ image        ┆ age_approx ┆ anatom_site_general ┆ lesion_id ┆ sex    │
 │ ---          ┆ ---        ┆ ---                 ┆ ---       ┆ ---    │
 │ str          ┆ i64        ┆ str                 ┆ str       ┆ str    │
 ╞══════════════╪════════════╪═════════════════════╪═══════════╪════════╡
 │ ISIC_0000000 ┆ 55         ┆ anterior torso      ┆ null      ┆ female │
 │ ISIC_0000001 ┆ 30         ┆ anterior torso      ┆ null      ┆ female │
 │ ISIC_0000002 ┆ 60         ┆ upper extremity     ┆ null      ┆ female │
 │ ISIC_0000003 ┆ 30         ┆ upper extremity     ┆ null      ┆ male   │
 │ ISIC_0000004 ┆ 80         ┆ posterior torso     ┆ null      ┆ male   │
 └──────────────┴────────────┴─────────────────────┴───────────┴────────┘,
 shape: (5, 10)
 ┌──────────────┬─────┬─────┬─────┬───┬─────┬──────┬─────┬─────┐
 │ image        ┆ MEL ┆ NV  ┆ BCC ┆ … ┆ DF  ┆ VASC ┆ SCC ┆ UNK │
 │ ---          ┆ --- ┆ 

In [6]:
df_labels[8]

image,MEL,NV,BCC,AK,BKL,DF,VASC,SCC,UNK
str,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""ISIC_0000009""",0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [7]:
# CANCER
df_labels.filter(pl.col("MEL") == 1)

image,MEL,NV,BCC,AK,BKL,DF,VASC,SCC,UNK
str,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""ISIC_0000002""",1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""ISIC_0000004""",1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""ISIC_0000013""",1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""ISIC_0000022_downsampled""",1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""ISIC_0000026_downsampled""",1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
…,…,…,…,…,…,…,…,…,…
"""ISIC_0073231""",1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""ISIC_0073237""",1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""ISIC_0073238""",1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [8]:
df_labels.shape[0]

25331

In [9]:
# Uh oh, no samples
df_labels.filter(pl.col("UNK") == 1)

image,MEL,NV,BCC,AK,BKL,DF,VASC,SCC,UNK
str,f64,f64,f64,f64,f64,f64,f64,f64,f64


In [10]:
# Drop "Unknown" column for training, no samples in dataset
df_labels = df_labels.drop(pl.col("UNK"))
df_labels.shape[0]
# Notice, no change in # of samples

25331

In [11]:
# Define train test split
train_size = 0.9
valid_size = 0.1

train_len = int(df_labels.shape[0] * 0.9)
valid_len = df_labels.shape[0] - train_len
is_train = ([True] * train_len) + ([False] * valid_len)
random.shuffle(is_train)
is_train[:10], len(is_train)

([False, True, True, True, True, True, True, True, True, True], 25331)

In [12]:
# Move data into folders for Pytorch Data Loader
image_path = os.path.join(path, "ISIC_2019_Training_Input", "ISIC_2019_Training_Input")
train_path = os.path.join(image_path, "train")
valid_path = os.path.join(image_path, "valid")
os.makedirs(train_path, exist_ok=True)
os.makedirs(valid_path, exist_ok=True)

for idx, image_name in enumerate(df_labels["image"]):
    folder = train_path if is_train[idx] else valid_path
    image = os.path.join(image_path, f"{image_name}.jpg")
    if os.path.exists(image):
        shutil.copy(image, os.path.join(folder, f"{image_name}.jpg"))

# idx = 0
# for label in df_labels.columns[1:]:
#     # make a directory for each label
#     os.makedirs(os.path.join(train_path, label), exist_ok=True)
#     os.makedirs(os.path.join(valid_path, label), exist_ok=True)
    
#     # move the corresponding images to that directory
#     for image_name in df_labels.filter(pl.col(label) == 1)["image"]:
#         folder = train_path if is_train[idx] else valid_path
#         image = os.path.join(image_path, f"{image_name}.jpg")
#         idx += 1 
        
#         if os.path.exists(image):
#             shutil.copy(image, os.path.join(folder, label, f"{image_name}.jpg"))

In [13]:
class ImageDataset(torch.utils.data.Dataset):
    def __init__(self, df_labels, img_dir, transform=None, target_transform=None):
        self.img_names = df_labels["image"]
        self.img_labels = df_labels.drop(pl.col("image")).to_torch().to(device)
        self.img_dir = img_dir
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.img_names[idx] + ".jpg")
        image = torchvision.io.read_image(img_path)
        label = self.img_labels[idx]
        if self.transform:
            image = self.transform(image)
        if self.target_transform:
            label = self.target_transform(label)
        return image, label

In [14]:
df = df_labels.with_columns(train = pl.Series(is_train))
df_train = df.filter(pl.col("train") == True).drop("train")
df_valid = df.filter(pl.col("train") == False).drop("train")
transforms = torchvision.models.VGG16_Weights.IMAGENET1K_V1.transforms()

train_ds = ImageDataset(df_train, train_path, transform=transforms)
# train_ds = torchvision.datasets.ImageFolder(train_path)
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=32, shuffle=True)

valid_ds = ImageDataset(df_valid, valid_path, transform=transforms)
# valid_ds = torchvision.datasets.ImageFolder(valid_path)
valid_loader = torch.utils.data.DataLoader(valid_ds, batch_size=32, shuffle=True)

transforms

ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)

In [15]:
model = torchvision.models.vgg16(weights=torchvision.models.VGG16_Weights.IMAGENET1K_V1).to(device)
model.classifier = nn.Sequential(
    nn.Linear(25088, 4096),
    nn.ReLU(),
    nn.Dropout(),
    nn.Linear(4096, 4096),
    nn.ReLU(),
    nn.Dropout(),
    nn.Linear(4096, 8),
).to(device)

In [16]:
metric = torchmetrics.Accuracy('multiclass', num_classes=8).to(device)
metric.reset()
for x_batch, y_batch in valid_loader:
    x_batch, y_batch = x_batch.to(device), y_batch.to(device)
    y_pred = model(x_batch)
    metric(y_pred, y_batch)
metric.compute().item()

0.0

In [24]:
def calc_accuracy(model: nn.Module, dataloader) -> float:
    metric = MulticlassAccuracy(num_classes=8)
    model.eval()  # set model to evaluation mode, disable certain features
    sum_ = 0
    count = 0
    with torch.no_grad():  # disable gradient calculations
        for X_batch, y_batch in dataloader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            z = model(X_batch)
            sum_ += (torch.argmax(z, dim=-1) == torch.argmax(y_batch, dim=-1)).sum().item()
            count += y_batch.shape[0]
    return sum_ / count

In [18]:
def train_model(
    model: nn.Module, 
    opt: torch.optim.Optimizer,
    dataloader: torch.utils.data.DataLoader,
    criterion: nn.Module = nn.CrossEntropyLoss(),
    epochs: int = 100,
):
    model.train()  # set model to training mode
    
    for epoch in range(epochs):
        avg_loss = 0
        for X_batch, y_batch in dataloader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            opt.zero_grad()
            z = model(X_batch)
            # print(f'{z}, {y_batch}')
            loss = criterion(z, y_batch)
            loss.backward()
            opt.step()
            avg_loss += loss.item()
            # print(loss)
        print(f'loss: {avg_loss / dataloader.batch_size}')

In [19]:
class Cnn(nn.Module):
    def __init__(self, in_channels=1, out_dim=30, im_width=96, device=torch.device("cpu")):
        super(Cnn, self).__init__()
        k, p, mode = 3, 1, "replicate"
        self.conv1 = nn.Conv2d(in_channels, 32, kernel_size=k, padding=p, padding_mode=mode)
        self.conv2 = nn.Conv2d(32, 32, kernel_size=k, padding=p, padding_mode=mode)           
        self.conv3 = nn.Conv2d(32, 64, kernel_size=k, padding=p, padding_mode=mode)       
        self.conv4 = nn.Conv2d(64, 64, kernel_size=k, padding=p, padding_mode=mode)       
        self.conv5 = nn.Conv2d(64, 128, kernel_size=k, padding=p, padding_mode=mode)       
        self.conv6 = nn.Conv2d(128, 128, kernel_size=k, padding=p, padding_mode=mode)
        self.pool = nn.MaxPool2d(2, 2)
        self.activate = nn.ReLU()
        self.flatten = nn.Flatten(start_dim=1)
        # self.fc1 = nn.Linear(128 * int(im_width / 8)**2, 4096)
        self.fc1 = nn.Linear(100352, 4096)
        self.fc2 = nn.Linear(4096, 2048)
        self.out = nn.Linear(2048, out_dim)

        # move the model
        self.device = device
        device = self.to(self.device)

    def forward(self, x):
                                          # 1, 96, 96
        x = self.activate(self.conv1(x))  # 32, 96, 96
        x = self.activate(self.conv2(x))  # 32, 96, 96
        x = self.pool(x)                  # 32, 48, 48
        x = self.activate(self.conv3(x))  # 64, 48, 48
        x = self.activate(self.conv4(x))  # 64, 48, 48
        x = self.pool(x)                  # 64, 24, 24
        x = self.activate(self.conv5(x))  # 128, 24, 24
        x = self.activate(self.conv6(x))  # 128, 24, 24
        x = self.pool(x)                  # 128, 12, 12
        x = self.flatten(x)               # 128 * 12 * 12
        x = self.activate(self.fc1(x))    # 4096
        x = self.activate(self.fc2(x))    # 2048
        return self.out(x)  # 30

In [20]:
lr = 1e-3
momentum = 0.9
epochs = 50

cnn = Cnn(in_channels=3, out_dim=8, im_width=256, device=device)
# cnn = torchvision.models.vgg16().to(device)
opt = torch.optim.SGD(cnn.parameters(), lr=lr, momentum=momentum)

start = time.time()
train_model(cnn, opt, train_loader, epochs=epochs)

loss: 33.09812680019632
loss: 29.183014794753316
loss: 27.730573913864784
loss: 26.782618586344125
loss: 26.000755775494163
loss: 25.35812479325554
loss: 24.73256476893207
loss: 24.222707221835748
loss: 23.702725953018877
loss: 23.116410920033022
loss: 22.714906851181432
loss: 22.210784119669878
loss: 21.569284515668418
loss: 20.95028918170683
loss: 20.237665034582093
loss: 19.295103372680288
loss: 18.348832711995776
loss: 17.097648390431377
loss: 15.43976806115059
loss: 13.641590184581053
loss: 11.4549738314156
loss: 9.253300251628913
loss: 7.327825180128559
loss: 5.842324339672621
loss: 4.572420750577732
loss: 3.5671364207978766
loss: 2.8061413781439213
loss: 2.1507538097023833
loss: 1.8050592543618693
loss: 1.6923675283104058
loss: 1.1905162222922188
loss: 0.9446547325827671
loss: 0.8974702034459644
loss: 0.983858697680739
loss: 0.6925650661539287
loss: 0.44987463062336647
loss: 0.3631269513713557
loss: 0.3188355535796836
loss: 0.3486856877440699
loss: 0.1749460877442456
loss: 0.338

In [21]:
start - time.time()

-6911.721928119659

In [25]:
calc_accuracy(cnn, train_loader), calc_accuracy(cnn, valid_loader)

(0.9999122691582226, 0.6724546172059984)

In [23]:
start - time.time()

-7025.123349905014